# PII Model Benchmark

Compare small transformer PII / NER models against Sanctum's Presidio
pipeline on the project's 12-document fixture corpus. Models are taken
from `resources/pii-evaluation-literature.md` section 3 ("Small
transformer models for PII").

**Style** is deliberately close to Microsoft's
[`presidio-research`](https://github.com/microsoft/presidio-research):

* one `BenchmarkCase` per model, loaded lazily
* predictions land on disk (`./bench_results/<model>.json`) so scoring
  is decoupled from inference — you can aggregate without reloading
  any model
* re-uses `tests.evaluation.scorer.EntityScorer` (overlap-matched,
  type-aware, macro-averaged) so the numbers here are directly
  comparable to `pytest -m evaluation`

## Running on a small machine

This laptop has **≈ 4.8 GB RAM**. Every model runs on CPU, one at a
time, and is explicitly released (`del` + `gc.collect()` + torch cache
clear) before the next one is loaded. The registry in the
configuration cell marks models as `default=True` only when they fit
comfortably (< 300 M params). The large checkpoints
(`gliner_large`, `obi/deid_roberta_i2b2`, `NuNER_Zero`) are included
but disabled — flip `enabled=True` if your machine has the headroom.

## What the benchmark does NOT do

* No fine-tuning. All models are evaluated as shipped.
* No token-level F1. We use Sanctum's overlap/type scorer (i2b2-style);
  see §1 of the literature doc for why.
* No licence filter. Commercially-blocked models (Piiranha, Isotonic)
  are included as **reference numbers** — do not ship them.

## 1. One-time install

Uncomment the cell below the first time you run this notebook. The
packages are intentionally **not** pinned into Sanctum's production
requirements — they exist only for this evaluation notebook and would
otherwise break the air-gap invariant (HuggingFace first-load pulls
weights over the network).

Disk footprint for the default model set is ≈ 3.5 GB in `~/.cache/huggingface/`.

In [1]:
# !pip install --quiet \
#     torch==2.4.* --index-url https://download.pytorch.org/whl/cpu
# !pip install --quiet \
#     'transformers>=4.44' 'gliner>=0.2.13' 'huggingface-hub>=0.24' \
#     'pandas>=2' 'matplotlib>=3.8' tqdm ipykernel

## 2. Imports and repo wiring

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import pandas as pd

# This notebook lives in a separate research repo (sanctum-research) that
# sits next to the main Sanctum checkout. REPO_ROOT is this research repo
# (used for storing bench_results); SANCTUM_ROOT is the sibling Sanctum
# checkout we import from and read fixtures from. Override with the env
# var SANCTUM_ROOT if your layout differs.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SANCTUM_ROOT = Path(os.environ.get("SANCTUM_ROOT", REPO_ROOT.parent / "sanctum")).resolve()
if not (SANCTUM_ROOT / "sanctum").is_dir():
    raise RuntimeError(
        f"Expected the Sanctum checkout at {SANCTUM_ROOT}. "
        f"Clone it next to this repo or set SANCTUM_ROOT."
    )
sys.path.insert(0, str(SANCTUM_ROOT))

from sanctum.core.models import DetectionResult  # type: ignore
from tests.evaluation.scorer import EntityScorer  # type: ignore

print(f"research root: {REPO_ROOT}")
print(f"sanctum root:  {SANCTUM_ROOT}")
print(f"python:        {sys.version.split()[0]}")

## 3. Configuration

Edit `BENCH_MODELS` below to enable/disable models. `enabled=False`
rows are skipped entirely — no weights are downloaded.

`SCORE_THRESHOLD` is applied to every transformer prediction before
scoring; it's the rough equivalent of Sanctum's analyzer threshold.
GLiNER's internal threshold is passed separately.

In [3]:
# Results directory — predictions land here as JSON, one file per model.
RESULTS_DIR = REPO_ROOT / "notebooks" / "bench_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Score cutoffs. Anything below is dropped before scoring.
SCORE_THRESHOLD = 0.35
GLINER_THRESHOLD = 0.40  # GLiNER scores are calibrated a bit differently

# Force CPU. Flip if you have a GPU and enough VRAM.
DEVICE = "cpu"

# Sanctum's taxonomy — target label set we score against.
SANCTUM_LABELS = [
    "PERSON",
    "LOCATION",
    "ORGANIZATION",
    "DATE_TIME",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "URL",
    "IP_ADDRESS",
    "IBAN_CODE",
    "US_SSN",
    "US_DRIVER_LICENSE",
    "US_BANK_NUMBER",
    "CREDIT_CARD",
]


@dataclass
class BenchCase:
    """A single model entry in the benchmark registry."""

    key: str                 # short id used for filenames + table rows
    hf_id: str               # HuggingFace model id (empty for the Sanctum baseline)
    family: str              # "sanctum" | "gliner" | "hf_token"
    params_m: float          # parameter count in millions (RAM estimate)
    licence: str
    notes: str
    enabled: bool = True


BENCH_MODELS: list[BenchCase] = [
    # --- Sanctum baseline (what the production CLI builds) --------------
    BenchCase(
        key="sanctum_presidio",
        hf_id="",
        family="sanctum",
        params_m=0.0,
        licence="—",
        notes="Presidio + spaCy en_core_web_lg (issue #7 fix: ORG enabled)",
    ),
    # --- GLiNER (zero-shot, Apache-2.0, airgap-friendly) ----------------
    BenchCase(
        key="gliner_small_v21",
        hf_id="urchade/gliner_small-v2.1",
        family="gliner",
        params_m=166,
        licence="Apache-2.0",
        notes="Smallest GLiNER; good CPU baseline",
    ),
    BenchCase(
        key="gliner_medium_v21",
        hf_id="urchade/gliner_medium-v2.1",
        family="gliner",
        params_m=209,
        licence="Apache-2.0",
        notes="Reported 0.81 F1 on CoNLL-2003 (ONNX)",
    ),
    BenchCase(
        key="gliner_multi_pii_v1",
        hf_id="urchade/gliner_multi_pii-v1",
        family="gliner",
        params_m=209,
        licence="Apache-2.0",
        notes="Top pick from literature §3 — PII-tuned, multilingual",
    ),
    BenchCase(
        key="gliner_large_v21",
        hf_id="urchade/gliner_large-v2.1",
        family="gliner",
        params_m=459,
        licence="Apache-2.0",
        notes="Too big for a 4.8 GB machine — enable only on >= 8 GB",
        enabled=False,
    ),
    # --- Clinical de-id (Presidio's TransformersNlpEngine candidates) ---
    BenchCase(
        key="stanford_deid_base",
        hf_id="StanfordAIMI/stanford-deidentifier-base",
        family="hf_token",
        params_m=110,
        licence="MIT",
        notes="PubMedBERT trained on radiology + i2b2 — clinical bias",
    ),
    BenchCase(
        key="obi_deid_roberta_i2b2",
        hf_id="obi/deid_roberta_i2b2",
        family="hf_token",
        params_m=355,
        licence="MIT",
        notes="RoBERTa-large on i2b2 2014; heavy on RAM",
        enabled=False,
    ),
    # --- PII-specialised (reference only — non-commercial licences) -----
    BenchCase(
        key="piiranha_v1",
        hf_id="iiiorg/piiranha-v1-detect-personal-information",
        family="hf_token",
        params_m=276,
        licence="CC-BY-NC-ND-4.0",
        notes="Reference number — not shippable. Reports weighted F1 0.93.",
    ),
    BenchCase(
        key="isotonic_deberta_ai4privacy",
        hf_id="Isotonic/deberta-v3-base_finetuned_ai4privacy_v2",
        family="hf_token",
        params_m=184,
        licence="CC-BY-NC-4.0",
        notes="Reference number — not shippable. Self-reports 0.98 on ai4privacy held-out.",
        enabled=False,
    ),
    BenchCase(
        key="lakshyakh93_deberta_pii",
        hf_id="lakshyakh93/deberta_finetuned_pii",
        family="hf_token",
        params_m=184,
        licence="MIT",
        notes="MIT-licensed PII fine-tune — benchmark the unknown",
    ),
    BenchCase(
        key="ab_ai_pii_model",
        hf_id="ab-ai/pii_model",
        family="hf_token",
        params_m=110,
        licence="Apache-2.0",
        notes="BERT-base-cased, self-reports F1 0.96 over 32 PII types",
    ),
]

pd.DataFrame([c.__dict__ for c in BENCH_MODELS])[
    ["key", "family", "params_m", "licence", "enabled", "notes"]
]

,key,family,params_m,licence,enabled,notes
0,sanctum_presidio,sanctum,0.0,—,True,Presidio + spaCy en_core_web_lg (issue #7 fix:...
1,gliner_small_v21,gliner,166.0,Apache-2.0,True,Smallest GLiNER; good CPU baseline
2,gliner_medium_v21,gliner,209.0,Apache-2.0,True,Reported 0.81 F1 on CoNLL-2003 (ONNX)
3,gliner_multi_pii_v1,gliner,209.0,Apache-2.0,True,"Top pick from literature §3 — PII-tuned, multi..."
4,gliner_large_v21,gliner,459.0,Apache-2.0,False,Too big for a 4.8 GB machine — enable only on ...
5,stanford_deid_base,hf_token,110.0,MIT,True,PubMedBERT trained on radiology + i2b2 — clini...
6,obi_deid_roberta_i2b2,hf_token,355.0,MIT,False,RoBERTa-large on i2b2 2014; heavy on RAM
7,piiranha_v1,hf_token,276.0,CC-BY-NC-ND-4.0,True,Reference number — not shippable. Reports weig...
8,isotonic_deberta_ai4privacy,hf_token,184.0,CC-BY-NC-4.0,False,Reference number — not shippable. Self-reports...
9,lakshyakh93_deberta_pii,hf_token,184.0,MIT,True,MIT-licensed PII fine-tune — benchmark the unk...


## 4. Load the corpus

Scans `tests/fixtures/**/*.{txt,json}` the same way the pytest evaluation
conftest does. Each document yields `(text, ground_truth_entities)`.

In [ ]:
FIXTURES_ROOT = SANCTUM_ROOT / "tests" / "fixtures"


def load_corpus() -> dict[str, dict]:
    """Return {doc_name: {\"text\": str, \"entities\": list[dict]}}."""
    corpus: dict[str, dict] = {}
    for json_path in sorted(FIXTURES_ROOT.rglob("*.json")):
        if json_path.name in {"schema.json", "MANIFEST.sha256.json"}:
            continue
        txt_path = json_path.with_suffix(".txt")
        if not txt_path.exists():
            continue
        ann = json.loads(json_path.read_text(encoding="utf-8"))
        corpus[ann.get("document", json_path.stem)] = {
            "text": txt_path.read_text(encoding="utf-8"),
            "entities": ann["entities"],
        }
    return corpus


CORPUS = load_corpus()
print(f"{len(CORPUS)} documents")
total_ents = sum(len(d["entities"]) for d in CORPUS.values())
print(f"{total_ents} ground-truth entities")

pd.DataFrame([
    {"document": name, "chars": len(d["text"]), "entities": len(d["entities"])}
    for name, d in CORPUS.items()
])

## 5. Label mapping

Each model family speaks its own label vocabulary. We map everything
into Sanctum's taxonomy (`SANCTUM_LABELS`) before scoring. GLiNER is
prompted with the Sanctum labels directly — the mapping is a round-trip
(lowercase prompt → uppercase Sanctum name).

For HF token-classification models, we maintain per-model maps below.
Unmapped labels are dropped.

Note: we do NOT try to undo every vendor's decomposition (e.g.
Piiranha splits PERSON into GIVENNAME + SURNAME). Two adjacent
predictions will each count as a TP because Sanctum's scorer is
overlap-based.

In [5]:
# GLiNER prompts — lowercase natural-language labels; better recall than uppercase enums.
GLINER_PROMPT_LABELS = [
    "person",
    "location",
    "organization",
    "date",
    "email address",
    "phone number",
    "url",
    "ip address",
    "iban",
    "social security number",
    "driver license number",
    "bank account number",
    "credit card number",
]
GLINER_TO_SANCTUM = dict(zip(GLINER_PROMPT_LABELS, SANCTUM_LABELS))

HF_LABEL_MAPS: dict[str, dict[str, str]] = {
    # Piiranha / ai4privacy taxonomy (17 classes). Sources: model card + ai4privacy/pii-masking-400k.
    "piiranha_v1": {
        "GIVENNAME": "PERSON",
        "SURNAME": "PERSON",
        "MIDDLENAME": "PERSON",
        "USERNAME": "PERSON",
        "EMAIL": "EMAIL_ADDRESS",
        "TELEPHONENUM": "PHONE_NUMBER",
        "CITY": "LOCATION",
        "STREET": "LOCATION",
        "BUILDINGNUM": "LOCATION",
        "ZIPCODE": "LOCATION",
        "DATEOFBIRTH": "DATE_TIME",
        "TIME": "DATE_TIME",
        "DATE": "DATE_TIME",
        "IDCARDNUM": "US_SSN",
        "SOCIALNUM": "US_SSN",
        "TAXNUM": "US_SSN",
        "DRIVERLICENSENUM": "US_DRIVER_LICENSE",
        "ACCOUNTNUM": "US_BANK_NUMBER",
        "CREDITCARDNUMBER": "CREDIT_CARD",
        "IBAN": "IBAN_CODE",
        "IP": "IP_ADDRESS",
        "URL": "URL",
    },
    "isotonic_deberta_ai4privacy": {
        # Same ai4privacy families but DeBERTa-v2 preserves uppercase naming
        "FIRSTNAME": "PERSON",
        "LASTNAME": "PERSON",
        "MIDDLENAME": "PERSON",
        "USERNAME": "PERSON",
        "EMAIL": "EMAIL_ADDRESS",
        "PHONE_NUMBER": "PHONE_NUMBER",
        "PHONEIMEI": "PHONE_NUMBER",
        "PHONENUMBER": "PHONE_NUMBER",
        "CITY": "LOCATION",
        "STREETADDRESS": "LOCATION",
        "ZIPCODE": "LOCATION",
        "COUNTRY": "LOCATION",
        "STATE": "LOCATION",
        "DATE": "DATE_TIME",
        "TIME": "DATE_TIME",
        "DOB": "DATE_TIME",
        "SSN": "US_SSN",
        "DRIVERLICENSE": "US_DRIVER_LICENSE",
        "BANKACCOUNTNUMBER": "US_BANK_NUMBER",
        "CREDITCARDNUMBER": "CREDIT_CARD",
        "IBAN": "IBAN_CODE",
        "IPV4": "IP_ADDRESS",
        "IPV6": "IP_ADDRESS",
        "URL": "URL",
    },
    "lakshyakh93_deberta_pii": {
        "FIRSTNAME": "PERSON",
        "LASTNAME": "PERSON",
        "MIDDLENAME": "PERSON",
        "EMAIL": "EMAIL_ADDRESS",
        "PHONE_NUMBER": "PHONE_NUMBER",
        "CITY": "LOCATION",
        "STREETADDRESS": "LOCATION",
        "STATE": "LOCATION",
        "DATE": "DATE_TIME",
        "SSN": "US_SSN",
        "IBAN": "IBAN_CODE",
        "IPV4": "IP_ADDRESS",
        "URL": "URL",
        "CREDITCARDNUMBER": "CREDIT_CARD",
    },
    "ab_ai_pii_model": {
        "PERSON": "PERSON",
        "EMAIL": "EMAIL_ADDRESS",
        "PHONENUMBER": "PHONE_NUMBER",
        "PHONE": "PHONE_NUMBER",
        "CITY": "LOCATION",
        "ADDRESS": "LOCATION",
        "COUNTRY": "LOCATION",
        "DATE": "DATE_TIME",
        "SSN": "US_SSN",
        "URL": "URL",
        "CREDITCARD": "CREDIT_CARD",
        "CREDITCARDNUMBER": "CREDIT_CARD",
    },
    # Stanford/obi clinical de-id — HIPAA Safe Harbor taxonomy. Organisation names of hospitals
    # and PATORG collapse onto ORGANIZATION; patients/doctors collapse onto PERSON.
    "stanford_deid_base": {
        "PATIENT": "PERSON",
        "STAFF": "PERSON",
        "HCW": "PERSON",
        "PERSON": "PERSON",
        "HOSPITAL": "ORGANIZATION",
        "HOSP": "ORGANIZATION",
        "PATORG": "ORGANIZATION",
        "ORG": "ORGANIZATION",
        "LOCATION-OTHER": "LOCATION",
        "CITY": "LOCATION",
        "STATE": "LOCATION",
        "COUNTRY": "LOCATION",
        "STREET": "LOCATION",
        "ZIP": "LOCATION",
        "AGE": "DATE_TIME",
        "DATE": "DATE_TIME",
        "PHONE": "PHONE_NUMBER",
        "EMAIL": "EMAIL_ADDRESS",
        "ID": "US_SSN",
        "URL": "URL",
    },
    "obi_deid_roberta_i2b2": {
        "PATIENT": "PERSON",
        "STAFF": "PERSON",
        "HCW": "PERSON",
        "HOSPITAL": "ORGANIZATION",
        "PATORG": "ORGANIZATION",
        "LOC": "LOCATION",
        "DATE": "DATE_TIME",
        "PHONE": "PHONE_NUMBER",
        "EMAIL": "EMAIL_ADDRESS",
        "ID": "US_SSN",
    },
}

## 6. Per-family predictors

Each `run_*` function takes a `BenchCase`, loads the corresponding
model, predicts on every document, releases the model, and returns a
dict of detections keyed by document name.

All three implementations end with the same cleanup block (`del`
locals → `gc.collect()` → torch cache clear). This is the key
memory-discipline trick: we never hold two models at once.

In [6]:
def _free_memory() -> None:
    """Drop GC refs and clear the torch allocator if torch is loaded."""
    gc.collect()
    try:
        import torch  # local import — torch may not be installed yet for the baseline-only run

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def _as_detection_dict(
    entity_type: str, start: int, end: int, score: float, text_span: str, recognizer: str
) -> dict:
    """Plain-dict form of DetectionResult — JSON serialisable."""
    return {
        "entity_type": entity_type,
        "start": int(start),
        "end": int(end),
        "score": float(score),
        "text_span": text_span,
        "context": "",
        "recognizer_name": recognizer,
    }

In [7]:
def run_sanctum(case: BenchCase, corpus: dict[str, dict]) -> dict[str, list[dict]]:
    """Sanctum baseline — builds the same engine the production CLI builds."""
    from sanctum.analyzer.adapter import PresidioAnalyzer  # type: ignore
    from sanctum.analyzer.nlp_config import create_nlp_engine  # type: ignore
    from sanctum.config.settings import settings  # type: ignore

    nlp = create_nlp_engine(model_name=settings.nlp.spacy_model)
    analyzer = PresidioAnalyzer(
        nlp_engine=nlp,
        default_score_threshold=SCORE_THRESHOLD,
    )

    predictions: dict[str, list[dict]] = {}
    for name, doc in corpus.items():
        preds = analyzer.analyze(doc["text"], score_threshold=SCORE_THRESHOLD)
        predictions[name] = [
            _as_detection_dict(p.entity_type, p.start, p.end, p.score, p.text_span, p.recognizer_name)
            for p in preds
        ]

    del analyzer, nlp
    _free_memory()
    return predictions

In [8]:
def run_gliner(case: BenchCase, corpus: dict[str, dict]) -> dict[str, list[dict]]:
    """GLiNER zero-shot predictor. Uses natural-language label prompts."""
    from gliner import GLiNER  # type: ignore

    model = GLiNER.from_pretrained(case.hf_id)
    try:
        model.to(DEVICE)
    except Exception:
        pass
    model.eval()

    predictions: dict[str, list[dict]] = {}
    for name, doc in corpus.items():
        text = doc["text"]
        # GLiNER has no chunker — long docs still work but get slow.
        raw = model.predict_entities(text, GLINER_PROMPT_LABELS, threshold=GLINER_THRESHOLD)
        mapped: list[dict] = []
        for r in raw:
            label = r["label"].lower()
            sanctum_label = GLINER_TO_SANCTUM.get(label)
            if not sanctum_label:
                continue
            mapped.append(
                _as_detection_dict(
                    sanctum_label,
                    r["start"],
                    r["end"],
                    r.get("score", 1.0),
                    r["text"],
                    case.key,
                )
            )
        predictions[name] = mapped

    del model
    _free_memory()
    return predictions

In [9]:
def run_hf_token(case: BenchCase, corpus: dict[str, dict]) -> dict[str, list[dict]]:
    """Generic HuggingFace token-classification predictor.

    Uses `aggregation_strategy="simple"` so BIO tags are merged into
    contiguous spans with a single score per span. Requires a per-model
    entry in `HF_LABEL_MAPS`.
    """
    import torch  # noqa: F401 — imported so _free_memory can clear the cache
    from transformers import (  # type: ignore
        AutoModelForTokenClassification,
        AutoTokenizer,
        pipeline,
    )

    label_map = HF_LABEL_MAPS.get(case.key, {})
    if not label_map:
        raise RuntimeError(f"No HF_LABEL_MAPS entry for {case.key}")

    tokenizer = AutoTokenizer.from_pretrained(case.hf_id)
    model = AutoModelForTokenClassification.from_pretrained(case.hf_id)
    ner = pipeline(
        "token-classification",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple",
        device=-1,  # -1 = CPU
    )

    predictions: dict[str, list[dict]] = {}
    try:
        for name, doc in corpus.items():
            text = doc["text"]
            raw = ner(text)
            mapped: list[dict] = []
            for r in raw:
                # Strip any BIO prefix just in case (pipeline usually does, but defence in depth).
                group = r.get("entity_group") or r.get("entity", "")
                group = group.split("-", 1)[-1].upper()
                sanctum_label = label_map.get(group)
                if not sanctum_label:
                    continue
                if float(r["score"]) < SCORE_THRESHOLD:
                    continue
                mapped.append(
                    _as_detection_dict(
                        sanctum_label,
                        r["start"],
                        r["end"],
                        r["score"],
                        r["word"],
                        case.key,
                    )
                )
            predictions[name] = mapped
    finally:
        del ner, model, tokenizer
        _free_memory()
    return predictions


FAMILY_RUNNERS: dict[str, Callable[[BenchCase, dict], dict[str, list[dict]]]] = {
    "sanctum": run_sanctum,
    "gliner": run_gliner,
    "hf_token": run_hf_token,
}

## 7. Run the benchmark

For each `enabled=True` case:

1. Skip if we already have `bench_results/<key>.json` on disk (delete
   the file to force a re-run).
2. Load the model, run it over every document, release it.
3. Write predictions + timing to `bench_results/<key>.json`.

Expect ≈ 2–8 minutes per model on a 4.8 GB laptop.

In [13]:
def _results_path(key: str) -> Path:
    return RESULTS_DIR / f"{key}.json"


def run_benchmark(cases: list[BenchCase], corpus: dict[str, dict], force: bool = False) -> None:
    for case in cases:
        if not case.enabled:
            print(f"[skip] {case.key} (enabled=False)")
            continue
        out_path = _results_path(case.key)
        if out_path.exists() and not force:
            print(f"[cache] {case.key} → {out_path.name}")
            continue
        runner = FAMILY_RUNNERS.get(case.family)
        if runner is None:
            print(f"[skip] {case.key} — no runner for family {case.family!r}")
            continue

        print(f"[run ] {case.key} ({case.family}, ~{case.params_m:.0f}M params)")
        t0 = time.perf_counter()
        try:
            predictions = runner(case, corpus)
        except Exception as exc:  # noqa: BLE001 — notebook wants to survive one-model failure
            print(f"       FAILED: {exc!r}")
            _free_memory()
            continue
        elapsed = time.perf_counter() - t0
        payload = {
            "case": case.__dict__,
            "elapsed_seconds": elapsed,
            "predictions": predictions,
        }
        out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        print(f"       done in {elapsed:.1f}s → {out_path.name}")


run_benchmark(BENCH_MODELS[3:], CORPUS)

[run ] gliner_multi_pii_v1 (gliner, ~209M params)


/home/filippo/projects/sanctum/.venv-bench/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/home/filippo/projects/sanctum/.venv-bench/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 399 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/filippo/projects/sanctum/.venv-bench/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 410 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


       done in 394.5s → gliner_multi_pii_v1.json
[skip] gliner_large_v21 (enabled=False)
[run ] stanford_deid_base (hf_token, ~110M params)


...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/555M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/555M [00:00<?, ?B/s]

       done in 166.8s → lakshyakh93_deberta_pii.json
[run ] ab_ai_pii_model (hf_token, ~110M params)
       FAILED: OSError('You are trying to access a gated repo.\nMake sure to have access to it at https://huggingface.co/ab-ai/pii_model.\n401 Client Error. (Request ID: Root=1-69e52ca9-1850e8e067c341860a79a597;c0bdf7c5-6362-4eb3-8355-c46063dbfbeb)\n\nCannot access gated repo for url https://huggingface.co/ab-ai/pii_model/resolve/main/config.json.\nAccess to model ab-ai/pii_model is restricted. You must have access to it and be authenticated to access it. Please log in.')


## 8. Score every saved run

Scoring is pure disk-I/O — no models are in RAM at this point. Results
get folded into a long-form DataFrame (`entity × document × model`) so
the same frame can drive per-entity, per-document, and per-model
summaries.

In [14]:
def _to_detection_results(raw: list[dict]) -> list[DetectionResult]:
    return [
        DetectionResult(
            entity_type=r["entity_type"],
            start=r["start"],
            end=r["end"],
            score=r["score"],
            text_span=r["text_span"],
            context=r.get("context", ""),
            recognizer_name=r.get("recognizer_name", ""),
        )
        for r in raw
    ]


def score_all() -> pd.DataFrame:
    scorer = EntityScorer()
    rows: list[dict] = []
    for result_file in sorted(RESULTS_DIR.glob("*.json")):
        payload = json.loads(result_file.read_text(encoding="utf-8"))
        case = payload["case"]
        for doc_name, doc_payload in CORPUS.items():
            preds = _to_detection_results(payload["predictions"].get(doc_name, []))
            report = scorer.score(preds, doc_payload["entities"])
            rows.append({
                "model": case["key"],
                "family": case["family"],
                "document": doc_name,
                "entity_type": "__overall__",
                "precision": report.overall.precision,
                "recall": report.overall.recall,
                "f1": report.overall.f1,
                "tp": report.overall.true_positives,
                "fp": report.overall.false_positives,
                "fn": report.overall.false_negatives,
            })
            for etype, metrics in report.per_entity.items():
                rows.append({
                    "model": case["key"],
                    "family": case["family"],
                    "document": doc_name,
                    "entity_type": etype,
                    "precision": metrics.precision,
                    "recall": metrics.recall,
                    "f1": metrics.f1,
                    "tp": metrics.true_positives,
                    "fp": metrics.false_positives,
                    "fn": metrics.false_negatives,
                })
    return pd.DataFrame(rows)


scores = score_all()
print(f"scored {scores['model'].nunique()} models × {scores['document'].nunique()} documents")
scores.head()

scored 7 models × 12 documents


,model,family,document,entity_type,precision,recall,f1,tp,fp,fn
0,gliner_medium_v21,gliner,attorney_client_email.txt,__overall__,1.0,1.0,1.0,16,0,0
1,gliner_medium_v21,gliner,attorney_client_email.txt,DATE_TIME,1.0,1.0,1.0,3,0,0
2,gliner_medium_v21,gliner,attorney_client_email.txt,EMAIL_ADDRESS,1.0,1.0,1.0,3,0,0
3,gliner_medium_v21,gliner,attorney_client_email.txt,ORGANIZATION,1.0,1.0,1.0,1,0,0
4,gliner_medium_v21,gliner,attorney_client_email.txt,PERSON,1.0,1.0,1.0,8,0,0


## 9. Per-model aggregate

Macro-average across documents and entity types — directly comparable
to the `pytest -m evaluation -s` table. Sorted by aggregate F1.

In [15]:
per_entity = scores[scores["entity_type"] != "__overall__"]

# Corpus-wide per-entity totals, then macro-average across entity types.
by_entity = (
    per_entity.groupby(["model", "entity_type"], as_index=False)
    .agg(tp=("tp", "sum"), fp=("fp", "sum"), fn=("fn", "sum"))
)
by_entity["precision"] = by_entity["tp"] / (by_entity["tp"] + by_entity["fp"]).replace(0, pd.NA)
by_entity["recall"] = by_entity["tp"] / (by_entity["tp"] + by_entity["fn"]).replace(0, pd.NA)
by_entity["f1"] = (
    2 * by_entity["precision"] * by_entity["recall"]
    / (by_entity["precision"] + by_entity["recall"]).replace(0, pd.NA)
)
by_entity = by_entity.fillna(0.0)

summary = (
    by_entity.groupby("model", as_index=False)
    .agg(
        macro_precision=("precision", "mean"),
        macro_recall=("recall", "mean"),
        macro_f1=("f1", "mean"),
        total_tp=("tp", "sum"),
        total_fp=("fp", "sum"),
        total_fn=("fn", "sum"),
    )
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
summary.round(3)

,model,macro_precision,macro_recall,macro_f1,total_tp,total_fp,total_fn
0,gliner_medium_v21,0.767748,0.843635,0.77923,136,45,18
1,gliner_multi_pii_v1,0.819455,0.75834,0.778919,118,10,36
2,sanctum_presidio,0.56873,0.730281,0.610761,132,136,22
3,gliner_small_v21,0.591997,0.544504,0.532904,115,35,39
4,lakshyakh93_deberta_pii,0.478776,0.692735,0.529409,105,124,49
5,piiranha_v1,0.555128,0.501185,0.468054,49,55,105
6,stanford_deid_base,0.131909,0.222863,0.139585,41,55,113


## 10. Per-entity heatmap

Where does each model shine vs collapse? F1 pivoted on
`entity_type × model`. Useful for spotting recall gaps (cold columns)
and over-specialised models (one hot row, everything else cold).

In [ ]:
import matplotlib.pyplot as plt

heatmap = (
    by_entity.pivot(index="entity_type", columns="model", values="f1")
    .reindex(index=SANCTUM_LABELS)
    .fillna(0.0)
    .astype(float)
)

fig, ax = plt.subplots(figsize=(1.1 * len(heatmap.columns) + 2, 0.4 * len(heatmap.index) + 2))
im = ax.imshow(heatmap.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        val = heatmap.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val < 0.55 else "black", fontsize=8)
ax.set_title("Per-entity F1 (overlap match) — higher is better")
fig.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
plt.show()

## 11. Per-document F1

Which documents are hardest? Groups the `__overall__` rows across
models — a cold row here means the corpus itself is hard for every
model (likely a labelling-taxonomy or dense-quasi-identifier issue),
not a model problem.

In [ ]:
doc_view = (
    scores[scores["entity_type"] == "__overall__"]
    .pivot(index="document", columns="model", values="f1")
    .round(3)
)
doc_view

## 12. Inspection: look at a model's mistakes

For qualitative analysis. Re-scores one document against one model and
prints the false positives and false negatives. Edit `inspect_model` /
`inspect_doc`.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

available_models = sorted(p.stem for p in RESULTS_DIR.glob("*.json"))
doc_names = list(CORPUS)

model_dd = widgets.Dropdown(
    options=available_models or ["<no results>"],
    description="model:",
    layout=widgets.Layout(width="420px"),
)
doc_dd = widgets.Dropdown(
    options=doc_names,
    description="document:",
    layout=widgets.Layout(width="420px"),
)
limit_slider = widgets.IntSlider(
    value=15, min=1, max=50, step=1, description="max rows:",
    continuous_update=False,
)
out = widgets.Output()


def _render(*_):
    with out:
        clear_output(wait=True)
        model_key = model_dd.value
        doc_name = doc_dd.value
        limit = limit_slider.value
        payload_path = _results_path(model_key)
        if not payload_path.exists():
            print(f"No results for {model_key} — run the benchmark cell first.")
            return
        payload = json.loads(payload_path.read_text(encoding="utf-8"))
        preds = _to_detection_results(payload["predictions"].get(doc_name, []))
        report = EntityScorer().score(preds, CORPUS[doc_name]["entities"])
        print(f"== {model_key} on {doc_name} ==")
        print(
            f"overall  P={report.overall.precision:.3f}"
            f"  R={report.overall.recall:.3f}"
            f"  F1={report.overall.f1:.3f}"
            f"  (tp={report.overall.true_positives},"
            f" fp={report.overall.false_positives},"
            f" fn={report.overall.false_negatives})"
        )
        print(f"\nfalse positives (model hallucinated) — showing up to {limit}:")
        fps = report.false_positives[:limit]
        if not fps:
            print("  (none)")
        for d in fps:
            print(f"  {d.entity_type:<18} {d.text_span!r}")
        print(f"\nfalse negatives (model missed) — showing up to {limit}:")
        fns = report.false_negatives[:limit]
        if not fns:
            print("  (none)")
        for d in fns:
            print(f"  {d['entity_type']:<18} {d['text']!r}")


for w in (model_dd, doc_dd, limit_slider):
    w.observe(_render, names="value")

_render()
display(widgets.VBox([widgets.HBox([model_dd, doc_dd]), limit_slider, out]))

## 13. Notes for interpretation

* **Licence is a hard filter.** Piiranha and Isotonic score well by
  design — they were fine-tuned on ai4privacy held-out splits very
  similar to this corpus. Their numbers are reference-only; they
  cannot ship in a commercial Sanctum build.
* **Clinical de-id models (Stanford, obi) will look weak on legal
  prose.** That is expected — their label set does not cover
  e.g. PHONE / EMAIL cleanly, and the text distribution they were
  trained on is discharge summaries and radiology reports.
* **Sanctum's regex recognizers are already at ceiling on EMAIL, IBAN,
  IP, SSN.** The transformer models have no advantage on those and
  often regress. Read the heatmap column-by-column: where does the
  transformer *actually* beat Presidio?
* **GLiNER `multi_pii-v1` is the commercially-safe headline.** If its
  F1 on PERSON / ORGANIZATION / LOCATION beats Presidio by the 5–15
  point margin the literature predicts, the natural next step is a
  GLiNER recognizer adapter inside `sanctum/analyzer/` that plugs in
  alongside the spaCy-based one.
* **Re-running one model only.** Delete the matching JSON in
  `notebooks/bench_results/` and re-run cell §7; every other model is
  loaded from cache.